# Day 13 — Topological Sort

A **topological order** of a directed graph lists its vertices so that every edge
`u -> v` puts `u` somewhere before `v`. Think "build order", "course schedule",
"which cells of this spreadsheet do I recompute first".

Such an order exists **iff** the graph is a DAG (directed acyclic graph), so
sorting the graph and proving it has no cycle are the same computation. Two
classic ways to do it, both O(V + E):

1. **Kahn's algorithm** — repeatedly take a vertex whose in-degree is 0.
2. **DFS post-order** — a vertex is emitted only after all of its successors are,
   so reversing the finish order gives a topological order.


## The example graph

Eight courses; an edge means "the tail is a prerequisite of the head".

In [1]:
from collections import defaultdict, deque
import heapq

COURSE_NAMES = {
    0: 'intro programming', 1: 'discrete math', 2: 'data structures',
    3: 'algorithms', 4: 'linear algebra', 5: 'databases',
    6: 'machine learning', 7: 'capstone',
}
N = 8
EDGES = [(0, 2), (0, 3), (1, 3), (1, 4), (2, 5), (3, 5),
         (3, 6), (4, 6), (5, 7), (6, 7)]


def build(n, edges):
    """Adjacency list + in-degree array for vertices 0..n-1."""
    adj = defaultdict(list)
    indeg = [0] * n
    for u, v in edges:
        adj[u].append(v)
        indeg[v] += 1
    return adj, indeg


adj, indeg = build(N, EDGES)
for u in range(N):
    print('%d %-20s in-degree %d  ->  %s'
          % (u, COURSE_NAMES[u], indeg[u], sorted(adj[u]) or '-'))

0 intro programming    in-degree 0  ->  [2, 3]
1 discrete math        in-degree 0  ->  [3, 4]
2 data structures      in-degree 1  ->  [5]
3 algorithms           in-degree 2  ->  [5, 6]
4 linear algebra       in-degree 1  ->  [6]
5 databases            in-degree 2  ->  [7]
6 machine learning     in-degree 2  ->  [7]
7 capstone             in-degree 2  ->  -


## 1. Kahn's algorithm

Keep a queue of vertices with in-degree 0 — the ones with nothing left blocking
them. Pop one, output it, and decrement the in-degree of each successor; any
successor that drops to 0 joins the queue.

If the loop ends before all `n` vertices came out, the leftovers are exactly the
vertices trapped in (or downstream of) a cycle.

In [2]:
def kahn(n, edges):
    """Topological order, or None if the graph has a cycle."""
    adj, indeg = build(n, edges)
    queue = deque(v for v in range(n) if indeg[v] == 0)
    order = []

    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1          # u is placed, so v has one fewer blocker
            if indeg[v] == 0:      # v is now free to go
                queue.append(v)

    return order if len(order) == n else None


order = kahn(N, EDGES)
print(order)
print(' -> '.join(COURSE_NAMES[v] for v in order))

[0, 1, 2, 3, 4, 5, 6, 7]
intro programming -> discrete math -> data structures -> algorithms -> linear algebra -> databases -> machine learning -> capstone


### Grouping it into rounds

Take *every* free vertex at once instead of one at a time and you get the
graph's levels. Everything inside one level is mutually independent, which is
exactly the set of tasks a build system, or any parallel task runner,
can dispatch at once. The number of levels is the length of the longest path,
i.e. the shortest possible schedule.

In [3]:
def kahn_levels(n, edges):
    adj, indeg = build(n, edges)
    frontier = [v for v in range(n) if indeg[v] == 0]
    levels, seen = [], 0

    while frontier:
        levels.append(sorted(frontier))
        seen += len(frontier)
        nxt = []
        for u in frontier:
            for v in adj[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    nxt.append(v)
        frontier = nxt

    return levels if seen == n else None


for i, level in enumerate(kahn_levels(N, EDGES)):
    print('round %d:' % i, [COURSE_NAMES[v] for v in level])

round 0: ['intro programming', 'discrete math']
round 1: ['data structures', 'algorithms', 'linear algebra']
round 2: ['databases', 'machine learning']
round 3: ['capstone']


### A common follow-up: the smallest order

"Return the lexicographically smallest topological order" is a favourite
interview twist. One character changes: a min-heap instead of a queue, so among
the currently free vertices we always take the smallest. That is Day 05's heap
showing up again, and it costs O(V log V + E).

In [4]:
def kahn_smallest(n, edges):
    adj, indeg = build(n, edges)
    heap = [v for v in range(n) if indeg[v] == 0]
    heapq.heapify(heap)
    order = []

    while heap:
        u = heapq.heappop(heap)
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                heapq.heappush(heap, v)

    return order if len(order) == n else None


print(kahn_smallest(N, EDGES))

[0, 1, 2, 3, 4, 5, 6, 7]


## 2. DFS post-order

Run a DFS. When a vertex has no unexplored outgoing edge left, everything
reachable from it is already finished, so appending it now guarantees it sits
*after* all of its successors in the finish list. Reverse that list at the end.

Cycle detection comes free if you use three colours: **white** = unseen,
**grey** = on the current recursion path, **black** = finished. Meeting a grey
vertex means we walked back onto our own path — a back edge, hence a cycle.

In [5]:
WHITE, GREY, BLACK = 0, 1, 2


def dfs_topo(n, edges):
    """Topological order via DFS finish times.  None on a cycle."""
    adj, _ = build(n, edges)
    color = [WHITE] * n
    finished = []

    def visit(u):
        color[u] = GREY
        for v in adj[u]:
            if color[v] == GREY:                    # back edge -> cycle
                return False
            if color[v] == WHITE and not visit(v):
                return False
        color[u] = BLACK
        finished.append(u)                          # successors already out
        return True

    for s in range(n):
        if color[s] == WHITE and not visit(s):
            return None

    return finished[::-1]


print('kahn :', kahn(N, EDGES))
print('dfs  :', dfs_topo(N, EDGES))
print('both are valid - a DAG usually has many topological orders')

kahn : [0, 1, 2, 3, 4, 5, 6, 7]
dfs  : [1, 4, 0, 3, 6, 2, 5, 7]
both are valid - a DAG usually has many topological orders


Python's recursion limit is about 1000, so a real graph wants the iterative
version. The trick is to keep an *iterator* per stack frame, so we resume where
we left off instead of rescanning the neighbour list.

In [6]:
def dfs_topo_iterative(n, edges):
    adj, _ = build(n, edges)
    color = [WHITE] * n
    finished = []

    for s in range(n):
        if color[s] != WHITE:
            continue
        stack = [(s, iter(adj[s]))]
        color[s] = GREY
        while stack:
            u, it = stack[-1]
            for v in it:
                if color[v] == GREY:
                    return None
                if color[v] == WHITE:
                    color[v] = GREY
                    stack.append((v, iter(adj[v])))
                    break
            else:                       # no break: u has no edge left
                color[u] = BLACK
                finished.append(u)
                stack.pop()

    return finished[::-1]


print(dfs_topo_iterative(N, EDGES))
print('same as recursive?', dfs_topo_iterative(N, EDGES) == dfs_topo(N, EDGES))

[1, 4, 0, 3, 6, 2, 5, 7]
same as recursive? True


## Checking the answer

Do not trust the algorithm — check the definition. Every edge must point
forward in the produced order.

In [7]:
def is_topological(order, n, edges):
    if order is None or sorted(order) != list(range(n)):
        return False
    pos = {v: i for i, v in enumerate(order)}
    return all(pos[u] < pos[v] for u, v in edges)


print(is_topological(kahn(N, EDGES), N, EDGES))
print(is_topological(dfs_topo(N, EDGES), N, EDGES))
print(is_topological([7, 6, 5, 4, 3, 2, 1, 0], N, EDGES))   # reversed: wrong

True
True
False


## When there is a cycle

Add one edge that makes the capstone a prerequisite of the intro course and
neither algorithm can produce anything. `None` is a poor error message though —
here is the version that hands back the actual cycle, which is what a build
tool or a scheduler should print.

In [8]:
def find_cycle(n, edges):
    """One cycle as a list of vertices, or None if the graph is a DAG."""
    adj, _ = build(n, edges)
    color = [WHITE] * n
    parent = [-1] * n
    cycle = []

    def visit(u):
        color[u] = GREY
        for v in adj[u]:
            if color[v] == GREY:            # walk the parent chain back to v
                x = u
                cycle.append(v)
                while x != v:
                    cycle.append(x)
                    x = parent[x]
                cycle.append(v)
                cycle.reverse()
                return False
            if color[v] == WHITE:
                parent[v] = u
                if not visit(v):
                    return False
        color[u] = BLACK
        return True

    for s in range(n):
        if color[s] == WHITE and not visit(s):
            return cycle
    return None


bad = EDGES + [(7, 0)]
print('kahn      :', kahn(N, bad))
print('dfs       :', dfs_topo(N, bad))
print('the cycle :', ' -> '.join(str(v) for v in find_cycle(N, bad)))
print('as courses:', ' -> '.join(COURSE_NAMES[v] for v in find_cycle(N, bad)))

kahn      : None
dfs       : None
the cycle : 0 -> 2 -> 5 -> 7 -> 0
as courses: intro programming -> data structures -> databases -> capstone -> intro programming


## LeetCode 207 / 210 — Course Schedule

LC 207 asks whether you can finish all courses; LC 210 asks for an order. Both
are one call to Kahn. The only thing to be careful about is the direction:
`prerequisites[i] = [a, b]` means *take b before a*, so the edge is `b -> a`.

In [9]:
def can_finish(num_courses, prerequisites):
    edges = [(b, a) for a, b in prerequisites]      # take b before a
    return kahn(num_courses, edges) is not None


def find_order(num_courses, prerequisites):
    edges = [(b, a) for a, b in prerequisites]
    return kahn(num_courses, edges) or []


print(can_finish(2, [[1, 0]]))
print(can_finish(2, [[1, 0], [0, 1]]))
print(find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))
print(find_order(2, [[1, 0], [0, 1]]))

True
False
[0, 1, 2, 3]
[]


## Complexity

| | Time | Space |
|---|---|---|
| Kahn | O(V + E) | O(V) |
| DFS post-order | O(V + E) | O(V) recursion / explicit stack |
| Lexicographically smallest | O(V log V + E) | O(V) |

Each vertex enters the queue once and each edge is relaxed once, so both are
linear. Kahn gives you the levels and detects a cycle by a length check; DFS
gives you the cycle itself and reuses the machinery from Day 12.

In [10]:
import random

assert is_topological(kahn(N, EDGES), N, EDGES)
assert is_topological(dfs_topo(N, EDGES), N, EDGES)
assert is_topological(dfs_topo_iterative(N, EDGES), N, EDGES)
assert kahn(N, EDGES + [(7, 0)]) is None
assert dfs_topo(N, EDGES + [(7, 0)]) is None
assert find_cycle(N, EDGES) is None
assert kahn(3, []) == [0, 1, 2]
assert find_order(2, [[1, 0], [0, 1]]) == []

random.seed(13)
for _ in range(200):                      # random DAGs: labels shuffled,
    n = random.randint(1, 12)             # edges always point forward
    perm = list(range(n))
    random.shuffle(perm)
    e = [(perm[i], perm[j])
         for i in range(n) for j in range(i + 1, n) if random.random() < .3]
    assert is_topological(kahn(n, e), n, e)
    assert is_topological(dfs_topo(n, e), n, e)
    assert is_topological(dfs_topo_iterative(n, e), n, e)

print('all assertions passed')

all assertions passed


## Where this goes next

Tomorrow is **Union-Find**, the other way to answer "are these two things
connected" — undirected, and without ever building an order. Kruskal's minimum
spanning tree algorithm then uses it directly.

Topological sort itself keeps showing up outside graph theory: a backward pass
in autograd is a reverse topological traversal of the computation graph, and any
scheduler that dispatches independent work in parallel is computing the levels
from `kahn_levels`.
